---
title: "Week 9 Activity — Teaching a Computer to Read Handwriting (MNIST)"
format: html
jupyter: python3
---

## What's this all about?

This unit has been about artificial intelligence — what it is, what it isn't, and the very specific trick at the heart of most of it: instead of *telling* a computer the rules for a task, we *show* it thousands of examples and let it figure out the rules itself. That's **machine learning**, and today you're going to do it. For real. With your own hands.

Our task is one of the most famous in the whole field: teaching a computer to read handwritten digits. You write a `7`, it says "that's a 7." Sounds trivial — until you realize *you* could never write down the rules for what makes a `7` a `7`. (Go ahead, try. "A horizontal line and then a diagonal"... except sometimes there's a little cross-bar, and everyone's handwriting is different, and...). You *know* a 7 when you see one, but you can't explain it. So instead, we'll show the computer 60,000 examples and let it learn.

The dataset is called **MNIST** — 70,000 little images of handwritten digits, collected from real people, that basically every AI researcher on Earth has practiced on. It's the "hello world" of machine learning.

> **Heads up:** this notebook trains an actual neural network, so one cell will take a minute or two to run. That's normal. You'll see it working through the examples. **This runs best on Google Colab**, where everything you need is already installed.

> **Running this:** click a gray code cell and press **Shift+Enter**. Run them in order, top to bottom.

> **A note on accessibility:** this activity is about *images*, so it could easily become a wall of pictures you can't read. It isn't built that way. Every digit is also drawn as a **text-picture** (a little shape made of typed characters), and every result is printed as a **table** that spells out `CORRECT` / `WRONG` in words instead of relying on color. If you use a screen reader, a Braille display, or screen magnification, you should be able to follow the whole thing from the text alone — the images are a bonus, not a requirement. (One honest caveat: the images themselves, like all charts in Colab, don't carry alt-text descriptions, so the printed text *is* your equivalent — it's not an afterthought.)

## Step 1: Get the data

We load the 70,000 images. They come already split into a **training set** (the examples the computer gets to learn from) and a **test set** (examples we hide away, to check whether it actually learned or just memorized).

In [ ]:
import tensorflow as tf
import matplotlib.pyplot as plt
import numpy as np

(train_images, train_labels), (test_images, test_labels) = tf.keras.datasets.mnist.load_data()

print("Training images:", train_images.shape[0])
print("Test images we'll hold back:", test_images.shape[0])
print("Each image is", train_images.shape[1], "by", train_images.shape[2], "pixels.")

## Step 2: Look at the data (remember — images are just numbers)

Back in this course we learned that an image is just a grid of numbers, each number a brightness. Here's the receipt. First a tiny bit of setup: two helpers we'll reuse all notebook. One turns a digit into a **text-picture** — a small shape made of typed characters, so the digit's form is something you can *read*, whether you see the screen, magnify it, or run a screen reader / Braille display. The other shows a digit two ways at once: as that text-picture (always) and as an image (for anyone who'd rather look).

In [ ]:
def to_text_picture(image, width=14):
    """Render a digit as a small picture made of text characters."""
    img = np.array(image, dtype=float)
    if img.max() > 1:                       # works for either 0-255 or 0-1 images
        img = img / 255.0
    ramp = " .:-=+*#%@"                      # space = dark, '@' = brightest ink
    step = img.shape[1] // width
    lines = []
    for r in range(0, img.shape[0], step):
        row = ""
        for c in range(0, img.shape[1], step):
            brightness = img[r:r + step, c:c + step].mean()
            row += ramp[int(brightness * (len(ramp) - 1))] * 2   # *2 so it isn't squished
        lines.append(row)
    return "\n".join(lines)

def show_digit(image, caption):
    """Print a digit as text (always) and draw it as an image (for those who want it)."""
    print(caption)
    print(to_text_picture(image))
    print()
    plt.figure(figsize=(2.5, 2.5))
    plt.imshow(image, cmap="gray")
    plt.title(caption)
    plt.axis("off")
    plt.show()

print("Helpers ready.")

Now let's pull one digit out and look at it every which way:

In [ ]:
one_digit = train_images[0]
its_label = train_labels[0]

show_digit(one_digit, f"A human labeled this digit a '{its_label}'.")

The text-picture and the image are the same shape — the brighter the character (`@` brightest, a blank space darkest), the brighter that patch of handwriting. And here is that *exact same digit* as the raw numbers the computer actually stores: a 28×28 grid, 0 (black) to 255 (white).

In [ ]:
print("The same digit, as numbers (0 = black, 255 = white):")
print(one_digit)

Text-picture, numbers, or image — it's all the same thing. To the computer, "reading handwriting" means finding patterns in a grid of numbers like this one. There is no magic; there's just a *lot* of numbers.

## Step 3: A quick bit of prep

Two small housekeeping steps the network likes: squish the brightness values from the 0–255 range down to 0–1, which helps the learning go smoothly. (You don't need to understand the math — just know we're tidying the numbers.)

In [ ]:
train_images = train_images / 255.0
test_images = test_images / 255.0
print("Done. Numbers now range from 0 to 1 instead of 0 to 255.")

## Step 4: Build the learner

Now we build the **neural network** — the thing that will do the learning. You don't need to follow every line. The shape of it is: take the 28×28 grid, run it through a couple of layers of "decision-making," and end with 10 outputs — one for each possible digit, 0 through 9. The network's job is to light up the right one.

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),   # unroll the grid into a row
    tf.keras.layers.Dense(128, activation="relu"),    # a layer of pattern-finders
    tf.keras.layers.Dense(10, activation="softmax"),  # one output per digit 0-9
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])

print("Network built. Right now it knows NOTHING — its guesses would be random.")

## Step 5: Train it (this is the slow one — give it a minute)

This is the actual learning. The network looks at the training images, makes guesses, gets told the right answers, and nudges itself to do better — over and over, across all 60,000 examples, three times through (three "epochs"). Watch the **accuracy** number climb as it learns.

In [ ]:
history = model.fit(train_images, train_labels, epochs=3)

Did you watch that accuracy number go up? It started out bad and got good — by *practicing*. Nobody wrote a single rule about what a `7` looks like. The network figured it out from examples, exactly like you did when you were five.

## Step 6: The real test

Accuracy on the training images is a little like grading students on the exact homework they already saw. The honest test is the data we **hid back in Step 1** — images the network has never laid eyes on. If it does well here, it genuinely *learned* the idea of digits rather than memorizing.

In [ ]:
test_loss, test_accuracy = model.evaluate(test_images, test_labels, verbose=0)
print(f"Accuracy on digits it has NEVER seen: {test_accuracy:.1%}")

## Step 7: Watch it make predictions

Let's point the trained network at the first ten test digits and see what it says. We'll print a **table** first — the model's guess next to the true answer, with `CORRECT` or `WRONG` spelled out in words (not just a color) so it reads cleanly however you're reading it.

In [ ]:
predictions = model.predict(test_images, verbose=0)
guesses = np.argmax(predictions, axis=1)

print(f"{'image #':>7}  {'true':>4}  {'model said':>10}  result")
print("-" * 37)
for i in range(10):
    truth = test_labels[i]
    guess = guesses[i]
    result = "CORRECT" if guess == truth else "WRONG  <-- slipped up here"
    print(f"{i:>7}  {truth:>4}  {guess:>10}  {result}")

Want to see them as well? This draws those same ten digits, each titled with the model's guess and a text marker — `OK` if it was right, `X` if it was wrong — so the verdict is in words, not only color:

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(test_images[i], cmap="gray")
    mark = "OK" if guesses[i] == test_labels[i] else "X"
    plt.title(f"says {guesses[i]} [{mark}]")
    plt.axis("off")
plt.tight_layout()
plt.show()

## Step 8: Find its mistakes

No AI is perfect, and the *mistakes* are the most interesting part — they show you how the thing "thinks." Let's hunt down some digits it got wrong and look at them. Often you'll find the network's errors are ones *you* might make too: a sloppy `4` that really does look like a `9`.

First, the mistakes as a **table** — every row is a digit the network botched, with what it guessed and what the digit actually was:

In [ ]:
wrong = np.where(guesses != test_labels)[0]

print(f"It got {len(wrong)} of the {len(test_labels)} test digits wrong.\n")
print(f"{'image #':>7}  {'model said':>10}  {'actually was':>12}")
print("-" * 34)
for i in wrong[:8]:
    print(f"{i:>7}  {guesses[i]:>10}  {test_labels[i]:>12}")

Now the interesting part: let's actually *look at* a few of those mistakes as text-pictures, so you can judge for yourself whether the network's confusion was reasonable. (These print as readable text shapes, then as images.)

In [ ]:
for i in wrong[:3]:
    show_digit(test_images[i], f"The model said {guesses[i]}, but a human called it {test_labels[i]}:")

Be honest: how many of these would *you* have gotten wrong too, if someone shoved that scrawl in front of you with no context? That's worth sitting with.

## Reflection

A sentence or two each (these go in your discussion post):

1. We never wrote a rule for what any digit "looks like" — the network learned from 60,000 examples. Now imagine the examples had been biased somehow (say, only handwriting from one country, or one age group). How might that show up in the finished system? Connect this to what we discussed about **bias in AI** this unit.
2. The system got something like 97% of digits right — but it's *confidently wrong* on the rest, with no idea it erred. Name a real-world use of handwriting (or image) recognition where being wrong 3% of the time, with full confidence, would actually matter. Who would it matter *to*?

## What to turn in

Post on the **Week 9 discussion thread**:

1. The test accuracy your network reached (the number from Step 6).
2. One of the network's **mistakes** from Step 8 — paste the text-picture, copy a row of the mistakes table, or drop in a screenshot, whatever's easiest for you — plus one sentence on whether *you* sympathize with that error.
3. Your two reflection answers.

Then reply to a classmate — did their network do better or worse than yours? (Everyone's will be a little different — that randomness is part of how the learning works.)

*Low-stakes — graded on completion. You just trained a neural network. Genuinely: put that on a résumé.*